# [LES - nut] PitzDaily

## Preamble

In [ ]:
# Standard Library
import sys
import os
from pathlib import Path
import foamnordic as fno
import numpy as np
import matplotlib.pyplot as plt
import onsaemiro as osm

# Machine Learning
from sklearn.ensemble import ExtraTreesRegressor, VotingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearnex import patch_sklearn
patch_sklearn()

Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


### Directory & Path

In [2]:
# FoamNordic Project Directory
PROJECT_DIR = Path("/scratch/<allocation-account>/<user>")
CASE_TYPE = "les"
CASE_NAME = "pitzDaily"
BASE_DIR = PROJECT_DIR / "Codes" / "FoamNordic"
MAIN_DIR = BASE_DIR / "foamnordic_tutorials" / "incompressible"
OF_SCRIPT_DIR = BASE_DIR / "openfoam_tutorials" / CASE_TYPE / CASE_NAME

# Output Directory
MODEL_DIR = MAIN_DIR / "model"
OUTPUT_DIR = MAIN_DIR / "output"

for directory in [MODEL_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

### Configuration

In [3]:
# DataGraph Configuration
FIG_X, FIG_Y = 3.5, 2.55
FIGURE_SIZE = (FIG_X, FIG_Y)
PALETTE = osm.get_palette("OKABE_ITO")
osm.set_style(figure_size=FIGURE_SIZE)

In [4]:
# HPC configuration
ACCOUNT = "<allocation-account>"
PARTITION = "small"
TIME = "00:15:00"
N_NODES = 1
N_TASKS = 1
CPUS_PER_TASK = 1
MEM_PER_CPU = "2G"

# FoamNordic Slurm configuration
scheduler = fno.Slurm(
    account=ACCOUNT,
    partition=PARTITION,
    time=TIME,
    nodes=N_NODES,
    ntasks=N_TASKS,
    cpus_per_task=CPUS_PER_TASK,
    mem_per_cpu=MEM_PER_CPU,
)

In [5]:
# FoamNordic configuration
SEED = 42
key = fno.Random.key(seed=SEED, scope="global")

## Example - Closure Modelling

### Smagorinsky Closure

In [6]:
# Smagorinsky model coefficients
C_K = 0.0265463553
C_E = 1.048

# Smagorinsky function for subgrid-scale turbulence modelling
def smagorinsky_function(velocity_grad, delta, C_k=C_K, C_e=C_E):
    strain_rate = fno.Math.symm(velocity_grad)
    strain_rate_trace = fno.Math.einsum("...ii->...", strain_rate)
    dev_strain_rate = fno.Math.dev(strain_rate)

    coefficient_a = C_e / delta
    coefficient_b = (2.0 / 3.0) * strain_rate_trace
    coefficient_c = 2.0 * C_k * delta * fno.Math.ddot(dev_strain_rate, strain_rate)

    discriminant = coefficient_b**2 + 4.0 * coefficient_a * coefficient_c
    sqrt_k = (-coefficient_b + fno.Math.sqrt(fno.Math.maximum(discriminant, 0.0))) / (2.0 * coefficient_a)
    sqrt_k = fno.Math.maximum(sqrt_k, 0.0)

    eddy_viscosity = C_k * delta * sqrt_k

    return eddy_viscosity

In [7]:
# Define the Smagorinsky closure
smagorinsky_closure = fno.Closure(
    name="nutFjord",
    operator=fno.Operator.function(smagorinsky_function),
    inputs={
        "velocity_grad": fno.Field.grad("U"),
        "delta": fno.Field.delta(),
    },
    outputs={
        "eddy_viscosity": fno.Field("nut"),
    },
)

### Case Definition

In [8]:
# Initialize the OpenFOAM case
case = fno.OpenFOAM.Case(
    name=CASE_NAME,
    case_dir=OF_SCRIPT_DIR,
    run_dir=OUTPUT_DIR,
    of_cmd="module load openfoam/2512",
    shell="bash",
    application="pimpleFoam",
)

case.initialize(ranks=N_TASKS, mesh="blockMesh", validate_mesh=True);

### Submit Job

In [9]:
# Connect the OpenFOAM case and SLURM scheduler (launch a longship instance)
longship = fno.Longship(case=case, closures=(smagorinsky_closure,))

# Set sail for the OpenFOAM case (submit the job to the HPC cluster)
run = longship.launch(start_timeout=900)

[FoamNordic] Preparing mesh with blockMesh: pitzDaily
[FoamNordic] Mesh is ready: pitzDaily
[FoamNordic] Sailing in background: pitzDaily


In [10]:
# Wait for the job to complete (polling the job status)
result = run.stop(force=False, timeout=3600, progress=True)

In [11]:
# Summary of the job result
result.summary(style="compact");

Job ID,Name,Status,Partition,Node,Elapsed
-,pitzDaily,succeeded,local,rc5183,00:00:56


### Postprocessing

In [12]:
# Postprocessing
post = result.postprocess

velocity = post.field("U", time_idx=-1)
pressure = post.field("p", time_idx=-1)

print("U shape:", velocity.shape)
print("p shape:", pressure.shape)

statistics = post.statistics(
    ["U", "p", "nut"],
    time_idx=-1,
    verbose=True,
)

U shape: (12225, 3)
p shape: (12225,)


Field,Min,Max,Mean,Std,RMS
U,2.180695e-02,1.395087e+01,6.371092e+00,2.649377e+00,6.900001e+00
p,-3.573660e+01,1.143900e+02,6.502952e+01,2.183933e+01,6.859880e+01
nut,2.761300e-10,1.352340e-04,2.312463e-06,5.793487e-06,6.237947e-06
